# Unit 04 - Metrics (Demo)

**Atoms served:** `U04-A5` (funnel), `U04-A6` (`OEC`), `U04-A7` (user/session definitions)

**Estimated runtime:** ~15 seconds

**After this notebook you can:** build a funnel table, show how `OEC` weights flip the winning variant, and see definitional choices change the answer before any statistics run.

## Without code

Read the printed tables only.

1. Funnel table: which stage drops most between control and treatment?
2. `OEC` table: with revenue-heavy weights treatment wins; with conversion-heavy weights control wins.
3. User vs session table: same raw clicks, different denominators, different `CTR`.

Same conclusions without executing cells.

## 1. The question

Two homepage variants were tested on an e-commerce site. Product wants a single number to decide the winner - an **Overall Evaluation Criterion (`OEC`)** from `V06` and `V26`.

Which variant wins depends on **weights**, **funnel stage**, and **what counts as a user**.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

We simulate 2,000 visitors split equally between control (`D_i=0`) and treatment (`D_i=1`). Treatment lifts add-to-cart but slightly hurts checkout completion.

In [ ]:
n = 2000
D = np.repeat([0, 1], n // 2)
np.random.seed(RANDOM_SEED)
view = np.ones(n)
browse = np.random.binomial(1, np.where(D==1, 0.72, 0.70))
cart = np.random.binomial(1, np.where(D==1, 0.40, 0.32) * browse)
checkout = np.random.binomial(1, np.where(D==1, 0.45, 0.62) * cart)
# treatment lifts order value when checkout happens, but checkout rate is lower
order_value = np.where(D==1, np.random.lognormal(mean=3.8, sigma=0.35, size=n),
                              np.random.lognormal(mean=3.4, sigma=0.35, size=n))
revenue = checkout * order_value

funnel = pd.DataFrame({'D_i': D, 'view': view, 'browse': browse, 'cart': cart, 'checkout': checkout, 'revenue': revenue})
print(funnel.groupby('D_i')[['view','browse','cart','checkout']].mean().round(3))

## 4. The naive move

Pick one late-funnel metric - checkout rate - and call the winner.

In [ ]:
checkout_rate = funnel.groupby('D_i')['checkout'].mean()
print('Checkout rate by arm:\n', checkout_rate.round(4))
winner_checkout = 'control' if checkout_rate[0] > checkout_rate[1] else 'treatment'
print('Winner on checkout alone:', winner_checkout)

Control wins on checkout because treatment trades late-stage completion for earlier engagement.

## 5. What actually happens

**Funnel stage by stage (`U04-A5`).** A win at browse can hide a loss at checkout.

In [ ]:
stages = ['browse','cart','checkout']
rates = funnel.groupby('D_i')[stages].mean()
lift = (rates.loc[1] - rates.loc[0]) / rates.loc[0]
summary = pd.DataFrame({'control': rates.loc[0], 'treatment': rates.loc[1], 'relative_lift': lift})
print(summary.round(4))

Treatment lifts cart sharply and hurts checkout; browse is a tie. Read the `relative_lift` column before you accept anyone's summary of "it won" - and notice that the browse difference we coded into the data vanished into noise at this sample size.

**`OEC` weights encode who owns the trade-off (`U04-A6`).** Change weights, change the winner.

In [ ]:
arm = funnel.groupby('D_i').agg(
    browse_rate=('browse','mean'), checkout_rate=('checkout','mean'), avg_revenue=('revenue','mean'))
# Every metric is scaled the same way - each divided by the best arm on that metric.
# Mixing raw rates with scaled revenue would let revenue decide every time.
scaled = arm / arm.max()

weights_biz = {'browse_rate': 0.1, 'checkout_rate': 0.2, 'avg_revenue': 0.7}
weights_conv = {'browse_rate': 0.1, 'checkout_rate': 0.7, 'avg_revenue': 0.2}

def oec_score(arm_id, w):
    row = scaled.loc[arm_id]
    return sum(w[k] * row[k] for k in w)

def winner(w):
    return 'treatment' if oec_score(1, w) > oec_score(0, w) else 'control'

print(arm.round(4))
print('Scaled to best arm per metric:\n', scaled.round(4))
print('Revenue-heavy OEC score - control:', round(oec_score(0, weights_biz), 4),
      'treatment:', round(oec_score(1, weights_biz), 4))
print('Revenue-heavy winner:', winner(weights_biz))
print('Conversion-heavy OEC score - control:', round(oec_score(0, weights_conv), 4),
      'treatment:', round(oec_score(1, weights_conv), 4))
print('Conversion-heavy winner:', winner(weights_conv))

Same data, same arms, two winners. Revenue-heavy weights favour treatment because it earns more per completed order; conversion-heavy weights favour control because it completes more checkouts. Nothing statistical separates these two answers - someone must own the weights, and that someone is a person, not a test.

**User vs session definitions (`U04-A7`, `V26`).** Counting sessions instead of users changes `CTR` before any test.

In [ ]:
# Same users, treatment users click more often -> more sessions
users = pd.DataFrame({
    'user_id': np.arange(500),
    'D_i': np.random.binomial(1, 0.5, 500)
})
users['sessions'] = np.where(users.D_i==1, np.random.poisson(2.2, 500), np.random.poisson(1.5, 500))
users['clicks'] = np.random.binomial(users['sessions'], np.where(users.D_i==1, 0.35, 0.30))

ctr_user = users.groupby('D_i').apply(lambda g: g.clicks.sum()/g.user_id.nunique())
ctr_session = users.groupby('D_i').apply(lambda g: g.clicks.sum()/g.sessions.sum())
print('CTR by user:', ctr_user.round(4).to_dict())
print('CTR by session:', ctr_session.round(4).to_dict())

## 6. What you do about it

- Map the business process to **funnel stages** before picking one number.
- Write down **`OEC` weights** and who approved them.
- Freeze **user, session, and window** definitions in the analysis plan.

**When this matters less:** Early exploration can tolerate shifting definitions; a ship decision cannot.

---

**Takeaway:** Metrics are design choices. Weights and definitions decide the winner before `p`-values enter the room.

**Back to the unit:** [V1](../V1/units/unit-04-hypothesis-and-metrics/README.md) · [V2](../V2/units/unit-04-hypothesis-and-metrics/README.md)